# Verificación del Entorno y Versiones de Librerías en Google Colab

Este notebook utiliza el mismo patrón de instalación `_pip(pkg, import_name)` implementado en los notebooks de las Fases 2.1 (SARIMAX) y 2.3 (TTM), sin fijar versiones de forma rígida.

Comprueba que todas las dependencias del proyecto se importen sin problemas y genera la salida solicitada:
> `Python X.X; TensorFlow/Keras X.X; scikit-learn X.X; statsmodels/pmdarima X.X; granite-tsfm/transformers X.X; entorno Google Colab; GPU NVIDIA T4; RAM disponible`

### 1. Instalación de dependencias con la función `_pip()` estándar del proyecto
Idéntica a la utilizada en `fase2_1_sarimax.ipynb` y `fase2_3_ttm.ipynb`:

In [ ]:
# Install dependencies not pre-installed in Colab (estilo oficial del proyecto)
import importlib
import subprocess
import sys

def _pip(pkg, import_name=None):
    if importlib.util.find_spec(import_name or pkg) is None:
        print(f"Installing {pkg}...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

# Modelado clásico (Fase 2.1)
_pip("pmdarima")
_pip("statsmodels")

# TTM & Foundation Models (Fase 2.3)
_pip("granite-tsfm[notebooks]", "tsfm_public")
_pip("transformers")
_pip("datasets")
_pip("accelerate")

# Adquisición de datos y utilidades (data_utils.py)
_pip("openmeteo-requests", "openmeteo_requests")
_pip("requests-cache", "requests_cache")
_pip("retry-requests", "retry_requests")
_pip("holidays")

importlib.invalidate_caches()
print("Dependencies ready.")

### 2. Verificación de importación de `pmdarima` y `granite-tsfm`

In [ ]:
# Verificación inmediata de pmdarima
try:
    import pmdarima
    print(f"✓ pmdarima importado con éxito (versión: {pmdarima.__version__})")
except Exception as e:
    print(f"✗ Aviso con pmdarima: {e}")
    print("  → En Google Colab, tras instalar pmdarima la primera vez, suele requerirse")
    print("    'Entorno de ejecución' > 'Reiniciar sesión' para que NumPy enlace los nuevos binarios C.")

# Verificación inmediata de granite-tsfm (tsfm_public)
try:
    import tsfm_public
    from tsfm_public.models.tinytimemixer import TinyTimeMixerForPrediction
    print("✓ granite-tsfm (tsfm_public) importado con éxito")
except Exception as e:
    print(f"✗ Error con tsfm_public: {e}")

### 3. Reporte de versiones del entorno y librerías

In [ ]:
import sys
import os
import platform
import importlib
import importlib.metadata

importlib.invalidate_caches()

def get_pkg_version(dist_name, import_name=None):
    """Obtiene la versión instalada mediante metadata o inspeccionando el módulo."""
    candidates = [dist_name, dist_name.replace("-", "_"), dist_name.replace("_", "-")]
    if import_name:
        candidates.extend([import_name, import_name.replace("-", "_")])
    for name in candidates:
        try:
            return importlib.metadata.version(name)
        except Exception:
            pass
    mod_name = import_name or dist_name.replace("-", "_")
    try:
        mod = importlib.import_module(mod_name)
        return getattr(mod, "__version__", getattr(mod, "VERSION", "instalado"))
    except ImportError:
        return "no instalado"
    except Exception as e:
        return f"error ({e.__class__.__name__})"

# --- 1. Detección del entorno Colab ---
try:
    import google.colab
    entorno = "Google Colab"
except ImportError:
    entorno = f"Local ({platform.system()} {platform.machine()})"

# --- 2. Detección de GPU ---
gpu_info = "Ninguna (CPU)"
try:
    import torch
    if torch.cuda.is_available():
        name = torch.cuda.get_device_name(0)
        vram = torch.cuda.get_device_properties(0).total_memory / (1024**3)
        gpu_info = f"{name} ({vram:.1f} GB VRAM)"
except Exception:
    pass

if gpu_info == "Ninguna (CPU)":
    try:
        import subprocess
        out = subprocess.check_output(
            ["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
            stderr=subprocess.DEVNULL,
            text=True
        ).strip()
        if out:
            gpu_info = out.splitlines()[0].replace(", ", " (") + ")"
    except Exception:
        pass

# --- 3. Detección de RAM Disponible y Total ---
ram_disp_gb = None
ram_tot_gb = None
try:
    import psutil
    mem = psutil.virtual_memory()
    ram_disp_gb = mem.available / (1024**3)
    ram_tot_gb = mem.total / (1024**3)
except Exception:
    try:
        with open("/proc/meminfo", "r") as f:
            lines = f.readlines()
        mem_dict = {l.split(":")[0].strip(): l.split(":")[1].strip() for l in lines if ":" in l}
        total_kb = float(mem_dict.get("MemTotal", "0 kB").split()[0])
        avail_kb = float(mem_dict.get("MemAvailable", "0 kB").split()[0])
        ram_disp_gb = avail_kb / (1024**2)
        ram_tot_gb = total_kb / (1024**2)
    except Exception:
        pass

if ram_disp_gb is not None and ram_tot_gb is not None:
    ram_str = f"{ram_disp_gb:.1f} GB / {ram_tot_gb:.1f} GB total"
elif ram_disp_gb is not None:
    ram_str = f"{ram_disp_gb:.1f} GB"
else:
    ram_str = "N/D"

# --- 4. Versiones de librerías del repositorio ---
py_ver       = platform.python_version()
tf_ver       = get_pkg_version("tensorflow")
keras_ver    = get_pkg_version("keras")
sklearn_ver  = get_pkg_version("scikit-learn", "sklearn")
sm_ver       = get_pkg_version("statsmodels")
pmd_ver      = get_pkg_version("pmdarima")
tsfm_ver     = get_pkg_version("granite-tsfm", "tsfm_public")
transf_ver   = get_pkg_version("transformers")
torch_ver    = get_pkg_version("torch")
pd_ver       = get_pkg_version("pandas")
np_ver       = get_pkg_version("numpy")
scipy_ver    = get_pkg_version("scipy")
plt_ver      = get_pkg_version("matplotlib")
sns_ver      = get_pkg_version("seaborn")
datasets_ver = get_pkg_version("datasets")
accel_ver    = get_pkg_version("accelerate")
req_ver      = get_pkg_version("requests")
om_ver       = get_pkg_version("openmeteo-requests", "openmeteo_requests")
req_cache_ver= get_pkg_version("requests-cache", "requests_cache")
retry_ver    = get_pkg_version("retry-requests", "retry_requests")
hf_hub_ver   = get_pkg_version("huggingface-hub", "huggingface_hub")
gapi_ver     = get_pkg_version("google-api-python-client", "googleapiclient")
holidays_ver = get_pkg_version("holidays")

# Cadena de salida principal solicitada:
summary_output = (
    f"Python {py_ver}; "
    f"TensorFlow/Keras {tf_ver}/{keras_ver}; "
    f"scikit-learn {sklearn_ver}; "
    f"statsmodels/pmdarima {sm_ver}/{pmd_ver}; "
    f"granite-tsfm/transformers {tsfm_ver}/{transf_ver}; "
    f"PyTorch {torch_ver}; "
    f"entorno {entorno}; "
    f"GPU {gpu_info}; "
    f"RAM disponible {ram_str}"
)

print("=" * 85)
print("OUTPUT SOLICITADO:")
print("=" * 85)
print(summary_output)
print("=" * 85)

# Tabla detallada
categories = [
    ("Entorno & Hardware", [
        ("Python", py_ver),
        ("Entorno", entorno),
        ("Sistema Operativo", f"{platform.system()} {platform.release()} ({platform.machine()})"),
        ("GPU asignada", gpu_info),
        ("RAM de sistema", ram_str),
    ]),
    ("Datos y Álgebra (data_utils.py & Fase 1 EDA)", [
        ("pandas", pd_ver),
        ("numpy", np_ver),
        ("scipy", scipy_ver),
    ]),
    ("Visualización (Fases 1, 2 y 3)", [
        ("matplotlib", plt_ver),
        ("seaborn", sns_ver),
    ]),
    ("Modelado Clásico (Fase 2.1 SARIMAX & Fase 3 Comparativa)", [
        ("scikit-learn (sklearn)", sklearn_ver),
        ("statsmodels", sm_ver),
        ("pmdarima", pmd_ver),
    ]),
    ("Deep Learning & Foundation Models (Fase 2.2 LSTM & Fase 2.3 TTM)", [
        ("tensorflow", tf_ver),
        ("keras", keras_ver),
        ("torch (PyTorch)", torch_ver),
        ("granite-tsfm (tsfm_public)", tsfm_ver),
        ("transformers", transf_ver),
        ("datasets", datasets_ver),
        ("accelerate", accel_ver),
    ]),
    ("Descarga de Datos & Cloud Sync (data_utils.py & Google Drive)", [
        ("requests", req_ver),
        ("openmeteo-requests", om_ver),
        ("requests-cache", req_cache_ver),
        ("retry-requests", retry_ver),
        ("huggingface-hub", hf_hub_ver),
        ("google-api-python-client", gapi_ver),
        ("holidays", holidays_ver),
    ]),
]

print("\nINVENTARIO DETALLADO DE LIBRERÍAS:")
print(f"{'Librería / Recurso':<35} | {'Versión / Estado':<45}")
print("-" * 85)
for cat_name, items in categories:
    print(f"\n[{cat_name}]")
    for name, ver in items:
        print(f"  {name:<33} | {ver:<45}")
print("=" * 85)
